<a href="https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing Lane 2: Refresh / Content Opportunity Scoring. The starter notebooks I already
ran are literally this lane's core exercise — a transparent hand rule (stale x visible) versus
a learned ranking, scored with Precision@K on a client-holdout split. That comparison already
showed a real, honest gap (see numbers below), so I know the signal exists in this data before
I commit seven weeks to it. This lane also matches the data I already understand best
(dim_content + fact_content_daily_performance / the starter CSV), and produces the artifact
type — a ranked review queue with reason codes — that maps directly to a real decision a
content team makes every week: what to look at first when they can't review everything.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/umairhussainn/ml-internship"
REPO_DIR = "ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # from work/notebooks/ back to repo root

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages in the starter slice")

30,000 pages in the starter slice


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Question: Out of a large content inventory, which pages should a reviewer look at first this
sprint, given they can only manually check a limited number?

Decision improved: prioritization of a fixed-capacity review queue (a content/SEO reviewer with
time for maybe 20-50 pages a week, not thousands).

Who acts: a human content reviewer or strategist, who reads the reason code, opens the page,
and decides to refresh, expand, protect, prune, or leave it — the model never edits anything
itself.

Cost of a wrong call:
- False positive (flagged, but fine): wastes a reviewer's limited time — opportunity cost, not
  direct harm, but too many of these erodes trust in the queue and reviewers start ignoring it.
- False negative (missed, but actually declining): a real losing page goes unreviewed for
  another cycle; for a high-demand page this compounds into real, ongoing traffic loss that's
  harder to recover the longer it's missed.
Because both errors cost something real, ranking (not just binary flagging) matters — the top
of the list needs to be trustworthy even if the tail is noisy.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
declining = df["trend_direction"].str.lower().eq("down")
declining_with_demand = declining & (df["impressions_90d"] >= 100)
low_ctr_visible = (df["impressions_90d"] >= 500) & df["avg_position"].between(0, 20, inclusive="right") & (df["ctr"] < 0.5)

print(f"Declining pages (raw label):              {declining.sum():,} ({declining.mean():.1%})")
print(f"Declining WITH real demand (>=100 impr.):  {declining_with_demand.sum():,} ({declining_with_demand.mean():.1%})")
print(f"Low-CTR, position 1-20, visible candidates: {low_ctr_visible.sum():,} ({low_ctr_visible.mean():.1%})")
print()
print("From the starter pipeline (outputs/model_report.md), client-holdout validated:")
print("  Hand-rule baseline  Precision@50 = 0.240")
print("  Random forest       Precision@50 = 0.740   (~3x lift)")

Declining pages (raw label):              16,262 (54.2%)
Declining WITH real demand (>=100 impr.):  13,152 (43.8%)
Low-CTR, position 1-20, visible candidates: 9,759 (32.5%)

From the starter pipeline (outputs/model_report.md), client-holdout validated:
  Hand-rule baseline  Precision@50 = 0.240
  Random forest       Precision@50 = 0.740   (~3x lift)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can say: this will be decision-support — a ranked, evidence-backed queue that helps a
reviewer spend limited time on the most promising pages first. Claims will be observed
(e.g. "these pages show declining trend with real demand") and directional ("the model ranks
higher-risk pages higher, on held-out clients"), backed by Precision@K on a client-holdout split.

What I cannot say: that a refresh causes recovery — that needs a real experiment, not this
data. I also can't claim I've found a Google ranking factor, and I won't treat the starter
label (trend_direction == "down") as ground truth — it's a proxy calculated from the current
window, not a real future outcome. Part of my plan over the next 7 weeks is to move toward a
stronger future-window label (prior 90 days of features -> next 30 days decline), which the
lane guide flags as the honest upgrade from the beginner proxy label I'm starting with.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.